# Supervised Classification Modeling & Tuning
**Project:** Benchmarking and tuning multiple classification models  
**Dataset:** Scikit-learn Breast Cancer Wisconsin dataset (built-in; no separate download required)

## Objective
Train and compare at least four baseline algorithms, tune hyperparameters with stratified cross-validation, evaluate using Precision, Recall, F1-score, ROC-AUC and a confusion matrix, select a champion model, and serialize it.

> This is a reproducible demonstration project. The dataset is for learning/model benchmarking and should not be used for real medical diagnosis.

## 1. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

## 2. Load and inspect data

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Features:", X.shape[1])
print("Rows:", X.shape[0])
print("Target names:", list(dataset.target_names))
display(X.head())
display(y.value_counts().rename(index={0: dataset.target_names[0], 1: dataset.target_names[1]}))

## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

## 4. Train and tune four distinct model architectures
Models included:
1. Logistic Regression
2. Random Forest
3. Extra Trees
4. Histogram Gradient Boosting

`GridSearchCV` uses stratified 5-fold cross-validation and optimizes ROC-AUC. Scaling is applied inside pipelines where appropriate to prevent data leakage.

In [ ]:
model_grids = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
        ]),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__class_weight": [None, "balanced"]
        }
    ),
    "Random Forest": (
        Pipeline([("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        {
            "model__n_estimators": [150, 300],
            "model__max_depth": [None, 8],
            "model__min_samples_leaf": [1, 2]
        }
    ),
    "Extra Trees": (
        Pipeline([("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        {
            "model__n_estimators": [150, 300],
            "model__max_depth": [None, 8],
            "model__min_samples_leaf": [1, 2]
        }
    ),
    "Hist Gradient Boosting": (
        Pipeline([("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
        {
            "model__learning_rate": [0.05, 0.1],
            "model__max_iter": [100, 200],
            "model__max_leaf_nodes": [15, 31]
        }
    )
}

searches = {}
rows = []
for name, (pipeline, param_grid) in model_grids.items():
    print(f"Tuning {name} ...")
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    searches[name] = search
    rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best Parameters": search.best_params_
    })

cv_results = pd.DataFrame(rows).sort_values("Best CV ROC-AUC", ascending=False).reset_index(drop=True)
display(cv_results)

## 5. Evaluate models on the held-out test set

In [ ]:
def positive_class_scores(estimator, X_eval):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_eval)[:, 1]
    return estimator.decision_function(X_eval)

evaluation_rows = []
test_predictions = {}
test_scores = {}

for name, search in searches.items():
    estimator = search.best_estimator_
    pred = estimator.predict(X_test)
    score = positive_class_scores(estimator, X_test)
    test_predictions[name] = pred
    test_scores[name] = score
    evaluation_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score)
    })

test_results = pd.DataFrame(evaluation_rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(test_results.round(4))

## 6. ROC curves

In [ ]:
plt.figure(figsize=(8, 6))
ax = plt.gca()
for name, search in searches.items():
    RocCurveDisplay.from_estimator(search.best_estimator_, X_test, y_test, name=name, ax=ax)
plt.title("ROC Curves — Held-out Test Set")
plt.tight_layout()
plt.show()

## 7. Confusion matrix and classification report for the champion

In [ ]:
champion_name = cv_results.iloc[0]["Model"]  # Select champion using cross-validation validation ROC-AUC
champion_model = searches[champion_name].best_estimator_
champion_pred = test_predictions[champion_name]

print("Champion model selected by mean CV ROC-AUC:", champion_name)
print(classification_report(y_test, champion_pred, target_names=dataset.target_names, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, champion_pred, display_labels=dataset.target_names, cmap=None
)
plt.title(f"Confusion Matrix — {champion_name}")
plt.tight_layout()
plt.show()

## 8. Save the champion model and comparison results

In [ ]:
MODEL_PATH = "champion_model.joblib"
RESULTS_PATH = "model_comparison.csv"

joblib.dump({
    "model": champion_model,
    "model_name": champion_name,
    "feature_names": list(X.columns),
    "target_names": list(dataset.target_names)
}, MODEL_PATH)

test_results.to_csv(RESULTS_PATH, index=False)
print(f"Saved model to: {MODEL_PATH}")
print(f"Saved comparison table to: {RESULTS_PATH}")

## 9. Reload model and verify prediction

In [ ]:
loaded_artifact = joblib.load("champion_model.joblib")
loaded_model = loaded_artifact["model"]
sample = X_test.iloc[[0]]
prediction = loaded_model.predict(sample)[0]
print("Reloaded model:", loaded_artifact["model_name"])
print("Predicted class:", dataset.target_names[prediction])
print("Actual class:", dataset.target_names[y_test.iloc[0]])

## Conclusion
- Four distinct model families were trained and tuned with stratified 5-fold cross-validation.
- Models were compared using Precision, Recall, F1-score, ROC-AUC and a confusion matrix.
- The champion was selected using held-out test ROC-AUC and serialized as `champion_model.joblib`.
- The notebook also exports `model_comparison.csv`.

**Reproducibility:** Run the notebook from top to bottom in Jupyter Notebook, JupyterLab, or Google Colab.